# AIMv2's Autoregressive Bet Against CLIP Orthodoxy

**Paper:** [https://arxiv.org/abs/2411.14402](https://arxiv.org/abs/2411.14402)  
**Authors:** Enrico Fini, Mustafa Shukor, Xiujun Li, Philipp Dufter, Michal Klein, David Haldimann, Sai Aitharaju, Victor Guilherme Turrisi da Costa, Louis Béthune, Zhe Gan, Alexander T Toshev, Marcin Eichner, Moin Nabi, Yinfei Yang, Joshua M. Susskind, Alaaeldin El-Nouby  
**Repository:** [https://github.com/apple/ml-aim](https://github.com/apple/ml-aim)  
**Framework:** pytorch  
**License:** NOASSERTION  

---

*Reproduction generated by Vivory Research — runs on free-tier hardware (Kaggle T4 / Oracle CPU / GitHub Actions).*
*Produced: 2026-05-06 13:16 UTC*


## 1. Setup

Install dependencies from the paper's `requirements.txt`. Some packages may need GPU-specific wheels — adjust for your Colab/Kaggle runtime.

In [ ]:
!pip install --quiet --upgrade pip
!pip install --quiet torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121


## 2. Repository

Clone the reference implementation.

In [ ]:
!git clone --depth 1 https://github.com/apple/ml-aim
%cd ml-aim
!ls -la


## 3. Dataset

Download the dataset. Replace this cell with the dataset-specific loading code from the repository's README or `scripts/download_data.sh`.

In [ ]:
!pip install --quiet datasets
from datasets import load_dataset
ds = load_dataset("huggingface/badges")
print(ds)


## 4. Configuration

Core hyperparameters. Consider reducing epochs/batch size to fit free-tier GPU limits (Kaggle T4: 16GB VRAM, 30h/week; Colab: variable).

In [ ]:
import os, json, random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Reduced for free-tier — adjust if you have more GPU budget.
CONFIG = {
    "seed": SEED,
    "max_epochs": 1,
    "batch_size": 16,
    "learning_rate": 1e-4,
    "subset_fraction": 0.1,  # use 10% of data for quick reproduction
}
print(json.dumps(CONFIG, indent=2))


## 5+6. Paper-aware evaluation (auto-generated)

The cell below was generated by Vivory's reproduction agent (Opus 4.7) from the paper's abstract, body, repo README, and claimed_metrics. It performs real measurement on a small subset and writes the result to `/kaggle/working/metrics.json` for the runner to ingest.

In [ ]:
!pip install -q transformers accelerate datasets

import os, json, torch, urllib.request
from transformers import AutoModel, AutoProcessor
from datasets import load_dataset

os.makedirs("/kaggle/working", exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_grad_enabled(False)

def write_unsupported(msg):
    print(f"[unsupported] {msg}")
    with open("/kaggle/working/metrics.json", "w") as f:
        json.dump({"unsupported_infrastructure": 1.0}, f)

# AIMv2-3B (89.5% paper number) needs frozen-trunk attentive probe training,
# which is infeasible in 30 min on T4. The LiT zero-shot variant is the
# closest official AIMv2 ImageNet evaluation runnable here.
model_id = "apple/aimv2-large-patch14-224-lit"
try:
    processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
    model = AutoModel.from_pretrained(
        model_id, trust_remote_code=True, torch_dtype=torch.float16
    ).to(device).eval()
except Exception as e:
    write_unsupported(f"could not load {model_id}: {e}")
    raise SystemExit(0)

# ImageNet-1k class names (standard public list)
try:
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/anishathalye/imagenet-simple-labels/master/imagenet-simple-labels.json",
        "/tmp/in_labels.json",
    )
    with open("/tmp/in_labels.json") as f:
        classnames = json.load(f)
    assert len(classnames) == 1000
except Exception as e:
    write_unsupported(f"class names fetch failed: {e}")
    raise SystemExit(0)

def encode_text(texts):
    inp = processor(text=texts, return_tensors="pt", padding=True, truncation=True).to(device)
    if hasattr(model, "get_text_features"):
        return model.get_text_features(**inp)
    out = model(**inp)
    return getattr(out, "text_embeds", None) or out[0].mean(1)

def encode_image(img):
    inp = processor(images=img, return_tensors="pt").to(device)
    if hasattr(model, "get_image_features"):
        return model.get_image_features(**inp)
    out = model(**inp)
    return getattr(out, "image_embeds", None) or out[0][:, 0]

# Build text classifier
prompts = [f"a photo of a {c}." for c in classnames]
feats = []
bs = 64
for i in range(0, len(prompts), bs):
    f = encode_text(prompts[i:i+bs]).float()
    feats.append(f / f.norm(dim=-1, keepdim=True))
text_features = torch.cat(feats, dim=0)

# Load ImageNet validation (try a few public mirrors)
ds = None
for name, split, img_key, lbl_key in [
    ("evanarlian/imagenet_1k_resized_256", "val", "image", "label"),
    ("imagenet-1k", "validation", "image", "label"),
    ("clip-benchmark/wds_imagenet1k", "test", "jpg", "cls"),
]:
    try:
        ds = load_dataset(name, split=split, streaming=True)
        IMG_KEY, LBL_KEY = img_key, lbl_key
        print(f"Loaded {name} [{split}]")
        break
    except Exception as e:
        print(f"skip {name}: {e}")

if ds is None:
    write_unsupported("no public ImageNet validation accessible")
    raise SystemExit(0)

N = 300
correct = 0
total = 0
for ex in ds:
    if total >= N:
        break
    try:
        img = ex[IMG_KEY]
        label = ex[LBL_KEY]
        if not hasattr(img, "convert"):
            continue
        img = img.convert("RGB")
        f = encode_image(img).float()
        f = f / f.norm(dim=-1, keepdim=True)
        logits = (f @ text_features.T)[0]
        pred = int(logits.argmax().item())
        if pred == int(label):
            correct += 1
        total += 1
        if total % 50 == 0:
            print(f"{total}: running acc = {100.0*correct/total:.2f}")
    except Exception as e:
        print(f"sample err: {e}")
        continue

if total == 0:
    write_unsupported("no samples evaluated")
else:
    acc = 100.0 * correct / total
    metrics = {"accuracy_imagenet": acc}
    with open("/kaggle/working/metrics.json", "w") as f:
        json.dump(metrics, f)
    print(f"Evaluated {total} samples; correct={correct}")
    print(metrics)

## Appendix — Reproduction policy

This notebook runs on **free-tier hardware only**:

- **Kaggle Notebooks** — T4 GPU, 30h/week quota
- **Oracle Cloud** — ARM 4-core CPU, no GPU
- **GitHub Actions** — 2-core CPU, no GPU, 6h timeout
- **Colab** — variable T4/V100, 12h sessions (manual only)

If the full experiment exceeds these limits, reduce `max_epochs` / `subset_fraction` in the config cell and note the delta in the reproduction report.
